In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")

# ==========================================
# Step 1: Data Preparation & Time-Series Split
# ==========================================
print("\n[Step 1] Data Preparation & Time-Series Split")

candidate_features = [
    'ImpPH', 'ImpPD', 'ImpPA',
    'HomeElo', 'AwayElo', 'EloDiff', 'EloHomeW', 'EloAwayW', 'EloDraw',
    'HomeOff', 'AwayOff', 'HomeDef', 'AwayDef',
    'HomeP', 'AwayP', 'PDiff',
    'HomeW', 'HomeD', 'AwayW', 'AwayD',
    'HomeHW', 'HomeHD', 'AwayAW', 'AwayAD',
    'AvgGDiff', 'AvgSDiff','AvgSTDiff', 'AvgCDiff', 'AvgFDiff',
    'HomeRestDays', 'AwayRestDays'
]

cat_features = ['HomeTeam', 'AwayTeam', 'Referee']

df = pd.read_csv('final_feature_engineered_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values(by='Date').reset_index(drop=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=candidate_features, inplace=True) 

split_date = pd.to_datetime('2024-07-01')
train_df = df[df['Date'] < split_date].copy()
test_df = df[df['Date'] >= split_date].copy()

target_col = 'FTR'
le = LabelEncoder()
y_train = le.fit_transform(train_df[target_col])
y_test = le.transform(test_df[target_col])

X_train_num = train_df[candidate_features]
X_test_num = test_df[candidate_features]

existing_cat_features = [c for c in cat_features if c in train_df.columns]
catboost_candidate_features = candidate_features + existing_cat_features

X_train_catboost = train_df[catboost_candidate_features]
X_test_catboost = test_df[catboost_candidate_features]
cat_features_idx = [X_train_catboost.columns.get_loc(c) for c in existing_cat_features]

print(f"Total Matches Train (2010 - 2023): {len(train_df)}")
print(f"Total Matches Test  (2024/25):     {len(test_df)}")

plt.figure(figsize=(8, 5))
split_counts = [len(train_df), len(test_df)]
plt.bar(['Training Set (2010-2023)', 'Test Set (2024/25)'], split_counts, color=['#4C72B0', '#55A868'])
plt.title('Time-Series Data Split Distribution', fontsize=14)
plt.ylabel('Number of Matches')
for i, v in enumerate(split_counts):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('step1_data_split.png', dpi=300)
plt.show() 

In [ ]:
train_df.head()

In [ ]:
# ==========================================
# Step 2: Feature Selection Methodologies
# ==========================================
print("\n[Step 2] Feature Selection Methodologies")

print("-> RFE ...")
rf_base = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf_base, n_features_to_select=15, step=1)
rfe.fit(X_train_num, y_train)
rfe_features = X_train_num.columns[rfe.support_].tolist()

# RFE feature importance visualization
rfe_importances = rfe.estimator_.feature_importances_
rfe_imp_df = pd.DataFrame({'Feature': rfe_features, 'Importance': rfe_importances}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(rfe_imp_df['Feature'], rfe_imp_df['Importance'], color='#5656E6')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.savefig('step2_rfe_importance.png', dpi=300)
plt.show()

print("-> PCA ...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

# get pca with all components to plot explained variance ratio
pca_full = PCA().fit(X_train_scaled)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# the bar plot for explained variance ratio and cumulative variance line
plt.figure(figsize=(10, 6))
components_x = np.arange(1, len(pca_full.explained_variance_ratio_) + 1)
# the bar plot for individual explained variance ratio
plt.bar(components_x, pca_full.explained_variance_ratio_, alpha=0.6, align='center', label='Individual Explained Variance', color='#4C72B0')
# the cumulative explained variance line
plt.plot(components_x, np.cumsum(pca_full.explained_variance_ratio_), marker='o', linestyle='-', color='#C44E52', label='Cumulative Explained Variance')
plt.axhline(y=0.95, color='black', linestyle='--', label='95% Variance Threshold')
plt.axvline(x=pca.n_components_, color='blue', linestyle=':', label=f'Selected Components: {pca.n_components_}')
plt.title('PCA Explained Variance Ratio & Cumulative Variance', fontsize=14)
plt.xlabel('Principal Component Index')
plt.ylabel('Variance Explained Ratio')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('step2_pca_variance.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# Step 3: Model Training & RPS Evaluation
# ==========================================
print("\n[Step 3] Model Training & Accuracy/RPS Evaluation (7 Models)")

def calc_rps(y_true, y_prob):
    n_cat = 3
    y_true_onehot = np.zeros((len(y_true), n_cat))
    y_true_onehot[np.arange(len(y_true)), y_true] = 1
    return np.mean(np.sum((np.cumsum(y_prob,1) - np.cumsum(y_true_onehot,1))**2, 1) / (n_cat-1))

catboost_rfe_features = rfe_features + existing_cat_features
cat_features_idx_for_rfe_catboost = [catboost_rfe_features.index(c) for c in existing_cat_features]

models_config = {
    'XGBoost':      (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='mlogloss'), rfe_features, 'numeric'),
    'RandomForest': (RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42), rfe_features, 'numeric'),
    'CatBoost':     (CatBoostClassifier(iterations=100, learning_rate=0.05, depth=5, random_state=42, verbose=0, cat_features=cat_features_idx_for_rfe_catboost), catboost_rfe_features, 'catboost'),
    'LogisticReg':  (LogisticRegression(max_iter=1000, random_state=42), None, 'pca'),
    'KNN':          (KNeighborsClassifier(n_neighbors=15), None, 'pca'),
    'ANN':          (MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42), None, 'pca'),
    'NaiveBayes':   (GaussianNB(), None, 'pca')
}

model_results = []
trained_models = {}
y_probs_dict = {}

for name, (clf, feats, mode) in models_config.items():
    if mode == 'pca':
        clf.fit(X_train_pca, y_train)
        y_pred = clf.predict(X_test_pca)
        y_prob = clf.predict_proba(X_test_pca)
    elif mode == 'numeric':
        clf.fit(X_train_num[feats], y_train)
        y_pred = clf.predict(X_test_num[feats])
        y_prob = clf.predict_proba(X_test_num[feats])
    elif mode == 'catboost':
        clf.fit(X_train_catboost[feats], y_train)
        y_pred = clf.predict(X_test_catboost[feats])
        y_prob = clf.predict_proba(X_test_catboost[feats])
        
    acc = accuracy_score(y_test, y_pred)
    rps = calc_rps(y_test, y_prob)
    model_results.append({'Model': name, 'Accuracy': acc, 'RPS': rps})
    trained_models[name] = clf
    y_probs_dict[name] = y_prob

res_df = pd.DataFrame(model_results).sort_values(by='RPS', ascending=True)

print("-" * 50)
print(f"{'Model':<15} | {'Accuracy':<10} | {'RPS':<10}")
print("-" * 50)
for _, row in res_df.iterrows():
    print(f"{row['Model']:<15} | {row['Accuracy']:.4f}     | {row['RPS']:.4f}")
print("-" * 50)

fig, ax1 = plt.subplots(figsize=(10, 6))
x = np.arange(len(res_df['Model']))
width = 0.5

ax1.bar(x, res_df['Accuracy'], width, label='Accuracy (Higher is Better)', color='#5656E6', alpha=0.7)
ax1.set_ylabel('Accuracy', color='#5656E6', fontsize=12)
ax1.set_ylim(0.40, 0.60)
ax1.tick_params(axis='y', labelcolor='#5656E6')
ax1.grid(True, axis='y', linestyle='--', alpha=0.7) 

ax2 = ax1.twinx()
ax2.plot(x, res_df['RPS'], label='RPS (Lower is Better)', color='#EF3435', marker='o', linewidth=2.5, markersize=10)
ax2.set_ylabel('Ranked Probability Score (RPS)', color='#EF3435', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#EF3435')
ax2.grid(False) 

ax1.set_xticks(x)
ax1.set_xticklabels(res_df['Model'], rotation=45, ha='right', fontsize=11)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.savefig('step3_model_comparison_optimized.png', dpi=300)
plt.show()

best_model_name = res_df.iloc[0]['Model']
best_clf = trained_models[best_model_name]
print(f"\n>> Best model: {best_model_name} (based on lowest RPS)")

In [ ]:
# ==========================================
# Step 4.5: Probability Calibration
# ==========================================
from sklearn.calibration import calibration_curve

print("\n Probability Calibration")

plt.figure(figsize=(10, 8))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly Calibrated") 


target_class_idx = 2 
colors = {'RandomForest': '#5656E6', 'XGBoost': '#FFA131', 'CatBoost': '#EF3435', 'NaiveBayes': '#b84074'}

for name in ['RandomForest', 'XGBoost', 'CatBoost', 'NaiveBayes']:
    if name in y_probs_dict:
       
        prob_pos = y_probs_dict[name][:, target_class_idx]
        
        fraction_of_positives, mean_predicted_value = calibration_curve(
            (y_test == target_class_idx).astype(int), 
            prob_pos, 
            n_bins=10
        )
        
        plt.plot(mean_predicted_value, fraction_of_positives, "s-", 
                 label=f"{name}", color=colors[name], markersize=5)

plt.ylabel("Fraction of Positives (Actual Win Rate)")
plt.xlabel("Mean Predicted Probability (Model Confidence)")
plt.title(f"Calibration Curve: Predicted vs. Actual Win Rate (Home Class)", fontsize=14)
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step4_5_calibration_curve.png', dpi=300)
plt.show()


In [ ]:
# ==========================================
# Step 4: Feature Importance of Best Model
# ==========================================
print("\n[Step 4] Extracting Feature Importance of Best Model")
if best_model_name in ['XGBoost', 'RandomForest', 'CatBoost']:
    if best_model_name == 'CatBoost':
         importances = best_clf.get_feature_importance()
         feats_for_importance = catboost_rfe_features
    else:
         importances = best_clf.feature_importances_
         feats_for_importance = rfe_features
         
    feat_imp = pd.DataFrame({'Feature': feats_for_importance, 'Importance': importances})
    feat_imp = feat_imp.sort_values(by='Importance', ascending=True)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='#5656E6')
    plt.title(f'Feature Importance ({best_model_name})', fontsize=14)
    plt.xlabel('Relative Importance')
    plt.tight_layout()
    plt.savefig('step4_feature_importance.png', dpi=300)
    plt.show() 

In [ ]:
# ==========================================
# Step 5: Benchmark Comparison
# ==========================================
print("\n[Step 5] Benchmark Comparison: Model RPS vs Bookmaker RPS")
raw_p_H = 1 / test_df['AvgH']
raw_p_D = 1 / test_df['AvgD']
raw_p_A = 1 / test_df['AvgA']
margin = raw_p_H + raw_p_D + raw_p_A

# normalized bookmaker probabilities (the formula in the prompt)
bookie_prob_H = raw_p_H / margin
bookie_prob_D = raw_p_D / margin
bookie_prob_A = raw_p_A / margin
y_prob_bookie = np.vstack((bookie_prob_A, bookie_prob_D, bookie_prob_H)).T 

bookie_rps = calc_rps(y_test, y_prob_bookie)
best_model_rps = res_df.iloc[0]['RPS']

plt.figure(figsize=(6, 5))
rps_vals = [bookie_rps, best_model_rps]
labels = ['Bookmaker (Market)', f'Best Model ({best_model_name})']
bars = plt.bar(labels, rps_vals, color=['#8C8C8C', '#55A868'], width=0.5)
plt.ylim(min(rps_vals)*0.95, max(rps_vals)*1.02)
plt.title('RPS Benchmark: Model vs Market (Lower is Better)', fontsize=14)
plt.ylabel('RPS Score')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.001, f"{yval:.5f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('step5_rps_benchmark.png', dpi=300)
plt.show() 

In [ ]:
# ==========================================
# Step 6: Edge vs Ratio vs Diff
# ==========================================
print("\n[Step 6] Value Definition: Edge vs Ratio vs Diff")

bt_df = test_df[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'AvgH', 'AvgD', 'AvgA']].copy().reset_index(drop=True)
outcomes = ['A', 'D', 'H']
best_probs = y_probs_dict[best_model_name]

bt_df['Bookie_A'] = bookie_prob_A.values if isinstance(bookie_prob_A, pd.Series) else bookie_prob_A
bt_df['Bookie_D'] = bookie_prob_D.values if isinstance(bookie_prob_D, pd.Series) else bookie_prob_D
bt_df['Bookie_H'] = bookie_prob_H.values if isinstance(bookie_prob_H, pd.Series) else bookie_prob_H

best_picks_idx = np.argmax(best_probs, axis=1)
bt_df['Pick'] = [outcomes[i] for i in best_picks_idx]
bt_df['Pick_Prob'] = np.max(best_probs, axis=1)
bt_df['Pick_Odds'] = bt_df.apply(lambda row: row[f'Avg{row["Pick"]}'], axis=1)
bt_df['Pick_Bookie'] = bt_df.apply(lambda row: row[f'Bookie_{row["Pick"]}'], axis=1)

# equation application for three different value formulas
bt_df['Edge'] = bt_df['Pick_Prob'] * bt_df['Pick_Odds'] - 1
bt_df['Ratio'] = bt_df['Pick_Odds'] / (1 / bt_df['Pick_Prob']) - 1
bt_df['Diff'] = bt_df['Pick_Prob'] - bt_df['Pick_Bookie']

def calculate_roi_at_threshold(df, metric_col, threshold, odds_col='Pick_Odds', unit=100):
    mask = df[metric_col] >= threshold
    bets = df[mask]
    if len(bets) == 0: return np.nan
    won = bets['Pick'] == bets['FTR']
    profit = np.where(won, unit * (bets[odds_col] - 1), -unit).sum()
    return (profit / (len(bets) * unit)) * 100

t_er = np.linspace(bt_df['Edge'].min(), bt_df['Edge'].max() - 0.01, 50)
t_df = np.linspace(bt_df['Diff'].min(), bt_df['Diff'].max() - 0.01, 50)

rois_edge = [calculate_roi_at_threshold(bt_df, 'Edge', t) for t in t_er]
rois_ratio = [calculate_roi_at_threshold(bt_df, 'Ratio', t) for t in t_er]
rois_diff = [calculate_roi_at_threshold(bt_df, 'Diff', t) for t in t_df]

plt.figure(figsize=(10, 5))
plt.plot(t_er, rois_edge, label='Edge (Prob * Odds - 1)', color='#5656E6', linewidth=4, linestyle='-')
plt.plot(t_er, rois_ratio, label='Ratio (Odds / Model_Odds - 1)', color='#FFA131', linewidth=2, linestyle='-.')
plt.plot(t_df, rois_diff, label='Diff (Prob - Norm_Bookie)', color='#EF3435', linewidth=3, linestyle='--')
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Threshold Value')
plt.ylabel('ROI (%)')
plt.legend(loc='best')
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('step6_value_formulas_comparison.png', dpi=300)
plt.show()



# ==========================================
# core backtesting function for Step 7, used for both Logic A and Logic B
# ==========================================
def execute_backtest(df, logic_name, pick_col, odds_col, prob_col, edge_col, threshold=0.02, initial_capital=10000, bet_sizing='flat', kelly_fraction=0.25, unit_bet=100):
    df_filtered = df[df[edge_col] >= threshold].copy()
    
    if len(df_filtered) == 0:
        return None, {'Logic': logic_name, 'Total Bets': 0, 'Win Rate (%)': 0, 'Avg Odds': 0, 'ROI (%)': 0, 'Max Drawdown (%)': 0, 'Final Capital ($)': initial_capital}
        
    df_filtered['Won'] = (df_filtered[pick_col] == df_filtered['FTR'])
    current_capital = initial_capital
    equity_curve = [initial_capital]
    
    for idx, row in df_filtered.iterrows():
        if bet_sizing == 'flat':
            bet_amount = unit_bet
        elif bet_sizing == 'kelly':
            b = row[odds_col] - 1  
            p = row[prob_col]         
            q = 1 - p
            kelly_f = (b * p - q) / b if b > 0 else 0
            bet_amount = current_capital * kelly_f * kelly_fraction if kelly_f > 0 else 0
            bet_amount = min(bet_amount, current_capital * 0.05) 
        
        if row['Won']:
            pnl = bet_amount * (row[odds_col] - 1)
        else:
            pnl = -bet_amount
            
        current_capital += pnl
        equity_curve.append(current_capital)
        df_filtered.at[idx, 'Bet_Amount'] = bet_amount
        df_filtered.at[idx, 'Capital'] = current_capital
        
    total_bets = len(df_filtered[df_filtered['Bet_Amount'] > 0])
    win_rate = df_filtered[df_filtered['Won']]['Won'].count() / total_bets if total_bets > 0 else 0
    profit = current_capital - initial_capital
    total_staked = df_filtered['Bet_Amount'].sum()
    roi = (profit / total_staked * 100) if total_staked > 0 else 0
    avg_odds = df_filtered[df_filtered['Bet_Amount'] > 0][odds_col].mean()
    
    peaks = pd.Series(equity_curve).cummax()
    drawdowns = (peaks - pd.Series(equity_curve)) / peaks
    mdd = drawdowns.max() * 100
    
    metrics = {
        'Logic': logic_name,
        'Total Bets': total_bets,
        'Win Rate (%)': win_rate * 100,
        'Avg Odds': avg_odds,
        'ROI (%)': roi,
        'Max Drawdown (%)': mdd,
        'Final Capital ($)': current_capital
    }
    return equity_curve, metrics

# ==========================================
# Step 7: Decision Mechanism Establishment (Logic A vs Logic B)
# ==========================================
print("\n[Step 7] Decision Mechanism Establishment: Maximum Likelihood (Logic A) vs Maximum Value (Logic B) [Using Flat Betting to Eliminate Capital Disturbance]")

edges = np.zeros_like(best_probs)
edges[:, 0] = best_probs[:, 0] - bt_df['Bookie_A']
edges[:, 1] = best_probs[:, 1] - bt_df['Bookie_D']
edges[:, 2] = best_probs[:, 2] - bt_df['Bookie_H']

# Logic A: Most Likely + Value
idx_A = np.argmax(best_probs, axis=1)
bt_df['Pick_LogicA'] = [outcomes[i] for i in idx_A]
bt_df['Prob_LogicA'] = np.max(best_probs, axis=1)
bt_df['Odds_LogicA'] = bt_df.apply(lambda row: row[f'Avg{row["Pick_LogicA"]}'], axis=1)
bt_df['Edge_LogicA'] = edges[np.arange(len(bt_df)), idx_A]

# Logic B: Max Diff
idx_B = np.argmax(edges, axis=1)
bt_df['Pick_LogicB'] = [outcomes[i] for i in idx_B]
bt_df['Edge_LogicB'] = np.max(edges, axis=1)
bt_df['Prob_LogicB'] = best_probs[np.arange(len(bt_df)), idx_B]
bt_df['Odds_LogicB'] = bt_df.apply(lambda row: row[f'Avg{row["Pick_LogicB"]}'], axis=1)

THRESHOLD = 0.02
curve_A_flat, metrics_A_flat = execute_backtest(bt_df, "Logic A: Most Likely + Value", 'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', THRESHOLD, bet_sizing='flat')
curve_B_flat, metrics_B_flat = execute_backtest(bt_df, "Logic B: Maximum Value (Max Diff)", 'Pick_LogicB', 'Odds_LogicB', 'Prob_LogicB', 'Edge_LogicB', THRESHOLD, bet_sizing='flat')

print("-" * 95)
print(f"{'Decision Logic':<35} | {'Bets':<5} | {'Avg Odds':<8} | {'Win Rate':<9} | {'Max DD':<9} | {'ROI':<8}")
print("-" * 95)
for m in [metrics_A_flat, metrics_B_flat]:
    print(f"{m['Logic']:<35} | {m['Total Bets']:<5} | {m['Avg Odds']:<8.2f} | {m['Win Rate (%)']:<8.1f}% | -{m['Max Drawdown (%)']:<8.1f}% | {m['ROI (%)']:<7.1f}%")
print("-" * 95)
print(">> conclusion: Logic A (Most Likely + Value) outperforms Logic B (Max Diff) in both ROI and risk metrics.")

# ==========================================
# Step 8: Experiment based on Logic A to find the Sweet Spot for Sports Betting
# ==========================================
print("\n[Step 8] sweet spot for sports betting: Ablation Study on Logic A with Hypothesis-Driven Filtering Strategies")

# five strategies
masks = {
    "Strategy 1 (Naive Value)": (bt_df['Edge_LogicA'] >= THRESHOLD),
    "Strategy 2 (Cap Longshots)": (bt_df['Edge_LogicA'] >= THRESHOLD) & (bt_df['Odds_LogicA'] <= 3.0),
    "Strategy 3 (Mid-Odds Value)": (bt_df['Edge_LogicA'] >= THRESHOLD) & (bt_df['Odds_LogicA'] >= 1.5) & (bt_df['Odds_LogicA'] <= 3.0),
    "Strategy 4 (Anti-Home Bias)": (bt_df['Edge_LogicA'] >= THRESHOLD) & (bt_df['Pick_LogicA'] != 'H'),
    "Strategy 5 (Golden Spot)": (bt_df['Edge_LogicA'] >= THRESHOLD) & (bt_df['Pick_LogicA'] != 'H') & (bt_df['Odds_LogicA'] >= 1.5) & (bt_df['Odds_LogicA'] <= 3.0)
}

ablation_metrics = []
for name, mask in masks.items():
    filtered_df = bt_df[mask].copy()
    _, m = execute_backtest(filtered_df, name, 'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', threshold=-999, bet_sizing='flat')
    ablation_metrics.append(m)

ss_df = pd.DataFrame(ablation_metrics)

print("-" * 105)
print(f"{'Ablation Strategy':<28} | {'Bets':<5} | {'Avg Odds':<8} | {'Win Rate':<9} | {'Max DD':<9} | {'ROI':<8} | {'Final Cap'}")
print("-" * 105)
for _, row in ss_df.iterrows():
    print(f"{row['Logic']:<28} | {row['Total Bets']:<5} | {row['Avg Odds']:<8.2f} | {row['Win Rate (%)']:<8.1f}% | -{row['Max Drawdown (%)']:<8.1f}% | {row['ROI (%)']:<7.1f}% | ${row['Final Capital ($)']:.2f}")
print("-" * 105)

# choose the best strategy based on ROI
best_strategy_row = ss_df.sort_values(by='ROI (%)', ascending=False).iloc[0]
best_strategy_name = best_strategy_row['Logic']
best_strategy_mask = masks[best_strategy_name]

# visualize the ROI of each strategy
plt.figure(figsize=(10, 6))
colors = ['#5656E6' if r > 0 else '#EF3435' for r in ss_df['ROI (%)']]
bars = plt.barh(ss_df['Logic'], ss_df['ROI (%)'], color=colors)
plt.axvline(0, color='black', linewidth=1.5)
plt.title(f'Step 8 Ablation Study: Performance of Hypothesis-Driven Strategies', fontsize=14)
plt.xlabel('ROI (%)')
for bar in bars:
    xval = bar.get_width()
    offset = 0.5 if xval > 0 else -3.5
    plt.text(xval + offset, bar.get_y() + bar.get_height()/2, f"{xval:+.1f}%", va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('step8_ablation_strategies.png', dpi=300)
plt.show()

print(f">> conclusion: [{best_strategy_name}] outperforms other strategies in the ablation study, demonstrating that mid-odds filtering can maximize returns.")

# ==========================================
# Step 9: Bankroll Management
# ==========================================
print(f"\n[Step 9] Bankroll Management Optimization: Applying the winning [{best_strategy_name}] betting pool to the Fractional Kelly engine...")

final_pool_df = bt_df[best_strategy_mask].copy()

# In the optimal betting pool, run flat betting and Kelly staking for comparison
curve_final_flat, metrics_final_flat = execute_backtest(final_pool_df, "Flat Betting (Fixed 100)", 'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', threshold=-999, bet_sizing='flat')
curve_final_kelly, metrics_final_kelly = execute_backtest(final_pool_df, "Fractional Kelly (1/4 Kelly)", 'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', threshold=-999, bet_sizing='kelly')

print("-" * 105)
print(f"{'Staking Plan':<28} | {'Bets':<5} | {'Avg Odds':<8} | {'Win Rate':<9} | {'Max DD':<9} | {'ROI':<8} | {'Final Cap'}")
print("-" * 105)
for m in [metrics_final_flat, metrics_final_kelly]:
    print(f"{m['Logic']:<28} | {m['Total Bets']:<5} | {m['Avg Odds']:<8.2f} | {m['Win Rate (%)']:<8.1f}% | -{m['Max Drawdown (%)']:<8.1f}% | {m['ROI (%)']:<7.1f}% | ${m['Final Capital ($)']:.2f}")
print("-" * 105)

plt.figure(figsize=(12, 6))
plt.plot(curve_final_flat, label=f"Flat Betting (ROI: {metrics_final_flat['ROI (%)']:+.1f}%)", color='#5656E6', linewidth=2.5)
plt.plot(curve_final_kelly, label=f"Fractional Kelly (ROI: {metrics_final_kelly['ROI (%)']:+.1f}%)", color='#EF3435', linewidth=2.5)
plt.axhline(10000, color='#FFA131', linestyle='--', alpha=0.7, label='Initial Capital ($10,000)')

plt.title(f'Step 9 Bankroll Growth: Kelly vs Flat (Applying {best_strategy_name})', fontsize=16)
plt.xlabel('Number of Bets Placed', fontsize=12)
plt.ylabel('Account Capital ($)', fontsize=12)
plt.legend(loc='best', fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('step9_bankroll_management.png', dpi=300)
plt.show()


In [ ]:
# ==========================================
# Step 10: Sensitivity Analysis of Kelly Fraction
# ==========================================
print(f"\n[Step 10] Sensitivity Analysis of Kelly Fraction: [{best_strategy_name}] Betting Pool")

final_pool_df = bt_df[best_strategy_mask].copy()

# define different Kelly fractions to test
fractions = [0.5, 0.25, 0.125] # corresponding to 1/2, 1/4, 1/8
fraction_labels = ['1/2 Kelly', '1/4 Kelly', '1/8 Kelly']
colors = ['#C44E52', '#EF3435', '#FFA131'] # colors represent risk levels

sensitivity_results = []
plt.figure(figsize=(12, 7))

# 1. baseline flat betting for reference
curve_flat, metrics_flat = execute_backtest(final_pool_df, "Flat Betting (Baseline)", 'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', threshold=-999, bet_sizing='flat')
plt.plot(curve_flat, label=f"Flat Betting (ROI: {metrics_flat['ROI (%)']:+.1f}%)", color='#5656E6', linewidth=2, linestyle='--')
sensitivity_results.append(metrics_flat)

# 2. loop through different Kelly fractions

for frac, label, color in zip(fractions, fraction_labels, colors):
    curve_k, metrics_k = execute_backtest(
        final_pool_df, 
        label, 
        'Pick_LogicA', 'Odds_LogicA', 'Prob_LogicA', 'Edge_LogicA', 
        threshold=-999, 
        bet_sizing='kelly', 
        kelly_fraction=frac
    )
    plt.plot(curve_k, label=f"{label} (ROI: {metrics_k['ROI (%)']:+.1f}%)", color=color, linewidth=2.5)
    sensitivity_results.append(metrics_k)


plt.axhline(10000, color='black', linestyle='-', alpha=0.3, label='Initial Capital ($10,000)')
plt.xlabel('Number of Bets Placed', fontsize=12)
plt.ylabel('Account Capital ($)', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step9_kelly_sensitivity_curve.png', dpi=300)
plt.show()

sens_df = pd.DataFrame(sensitivity_results)
print("\n" + "-" * 105)
print(f"{'Staking Strategy':<28} | {'Bets':<5} | {'Win Rate':<9} | {'Max DD':<9} | {'ROI':<8} | {'Final Capital'}")
print("-" * 105)
for _, row in sens_df.iterrows():
    print(f"{row['Logic']:<28} | {row['Total Bets']:<5} | {row['Win Rate (%)']:<8.1f}% | -{row['Max Drawdown (%)']:<8.1f}% | {row['ROI (%)']:<7.1f}% | ${row['Final Capital ($)']:.2f}")
print("-" * 105)

In [ ]:
# ==========================================
# Step 10.5: Avg Odds vs Max Odds Comparison
# ==========================================
print("\n[Step 10.5] Value Definition and Execution Efficiency Comparison: Avg Odds vs Max Odds")

# 1. reset index to ensure clean DataFrame for backtesting
bt_df = test_df.copy().reset_index(drop=True)
outcomes = ['A', 'D', 'H']
best_probs = y_probs_dict[best_model_name]

# 2. extract odds and normalized probabilities (using .values to ensure index is removed)
bt_df['Bookie_A'] = bookie_prob_A.values if hasattr(bookie_prob_A, 'values') else bookie_prob_A
bt_df['Bookie_D'] = bookie_prob_D.values if hasattr(bookie_prob_D, 'values') else bookie_prob_D
bt_df['Bookie_H'] = bookie_prob_H.values if hasattr(bookie_prob_H, 'values') else bookie_prob_H

# 3. determine the model's predicted choices
best_picks_idx = np.argmax(best_probs, axis=1)
bt_df['Pick'] = [outcomes[i] for i in best_picks_idx]
bt_df['Pick_Prob'] = np.max(best_probs, axis=1)

# 4. extract average odds and maximum odds (core comparison items)
bt_df['Avg_Odds'] = bt_df.apply(lambda row: row[f'Avg{row["Pick"]}'], axis=1)
bt_df['Max_Odds'] = bt_df.apply(lambda row: row[f'Max{row["Pick"]}'], axis=1)
bt_df['Pick_Bookie'] = bt_df.apply(lambda row: row[f'Bookie_{row["Pick"]}'], axis=1)

# 5. define value difference (Diff)
bt_df['Diff'] = bt_df['Pick_Prob'] - bt_df['Pick_Bookie']

# 6. run comparison experiments
# scenario A: use average odds (conservative benchmark, simulating market consensus execution)
_, metrics_avg = execute_backtest(bt_df, "Avg Odds (Market Consensus)", 
                                  'Pick', 'Avg_Odds', 'Pick_Prob', 'Diff', 
                                  threshold=0.02, bet_sizing='flat')

# scenario B: use maximum odds (ideal execution, simulating best-case execution)
_, metrics_max = execute_backtest(bt_df, "Max Odds (Best Execution)", 
                                  'Pick', 'Max_Odds', 'Pick_Prob', 'Diff', 
                                  threshold=0.02, bet_sizing='flat')

print("-" * 85)
print(f"{'Execution Mode':<30} | {'Avg Odds':<10} | {'ROI (%)':<10} | {'Final Cap'}")
print("-" * 85)
for m in [metrics_avg, metrics_max]:
    print(f"{m['Logic']:<30} | {m['Avg Odds']:<10.2f} | {m['ROI (%)']:<10.1f}% | ${m['Final Capital ($)']:.2f}")
print("-" * 85)

In [ ]:
# ==========================================
# Step 11: Season-by-Season Walk-Forward
# ==========================================
print("\n[Step 11] Season-by-Season Walk-Forward: Simulating Dynamic Retraining and Investing")


# 1. 2010-2017 as training, 2018-2024 as testing (rolling forward each season)
test_years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
rolling_summary = []
cumulative_equity = [10000] # initial capital for rolling backtest
outcomes = ['A', 'D', 'H']

# 2. for each season, retrain the model on all previous seasons and test on the current season
for year in test_years:
    print(f"\n>>> Ongoing: Training (2010-{year-1}) -> Prediction ({year} Season)")
    
    # data split for walk-forward
    train_df_wf = df[df['Date'].dt.year < year].copy()
    test_df_wf = df[df['Date'].dt.year == year].copy()
    
    if len(test_df_wf) == 0: continue
    
    # feature engineering and preprocessing (same as before, but done separately for each walk-forward iteration)
    le_wf = LabelEncoder()
    y_train_wf = le_wf.fit_transform(train_df_wf['FTR'])
    y_test_wf = le_wf.transform(test_df_wf['FTR'])
    
    scaler_wf = StandardScaler()
    X_train_wf = scaler_wf.fit_transform(train_df_wf[candidate_features])
    X_test_wf = scaler_wf.transform(test_df_wf[candidate_features])
    
    # RFE feature selection (using the same number of features for consistency)
    rf_base_wf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rfe_wf = RFE(estimator=rf_base_wf, n_features_to_select=15, step=1)
    rfe_wf.fit(X_train_wf, y_train_wf)
    
    # 15 features selected by RFE for this walk-forward iteration
    X_train_rfe = X_train_wf[:, rfe_wf.support_]
    X_test_rfe = X_test_wf[:, rfe_wf.support_]
    
    model_wf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    model_wf.fit(X_train_rfe, y_train_wf)
    probs_wf = model_wf.predict_proba(X_test_rfe)
    
    # 3. construct the betting pool for the current season based on the model's predictions and bookmaker odds
    season_bt = test_df_wf[['FTR', 'AvgH', 'AvgD', 'AvgA']].copy().reset_index(drop=True)
    picks_idx = np.argmax(probs_wf, axis=1)
    season_bt['Pick'] = [outcomes[i] for i in picks_idx]
    season_bt['Prob'] = np.max(probs_wf, axis=1)
    season_bt['Avg_Odds'] = season_bt.apply(lambda row: row[f'Avg{row["Pick"]}'], axis=1)
    
    # calculate normalized bookmaker probabilities for the current season
    margin = (1/season_bt['AvgH'] + 1/season_bt['AvgD'] + 1/season_bt['AvgA'])
    season_bt['Bookie_Prob'] = (1/season_bt.apply(lambda row: row[f'Avg{row["Pick"]}'], axis=1)) / margin
    season_bt['Diff'] = season_bt['Prob'] - season_bt['Bookie_Prob']
    
    # anti-home bias filtering for the current season (similar to Strategy 4 in Step 8)
    s4_mask = (season_bt['Diff'] >= 0.02) & (season_bt['Pick'] != 'H')
    season_pool = season_bt[s4_mask].copy()
    
    # 4. backtesting
    equity_wf, metrics_wf = execute_backtest(
        season_pool, f"Season {year}", 'Pick', 'Avg_Odds', 'Prob', 'Diff',
        threshold=-999, initial_capital=cumulative_equity[-1], bet_sizing='flat'
    )
    
    # record the metrics for this season and append to the rolling summary
    rolling_summary.append({
        'Season': year,
        'Bets': metrics_wf['Total Bets'],
        'WinRate': metrics_wf['Win Rate (%)'],
        'ROI': metrics_wf['ROI (%)'],
        'MDD': metrics_wf['Max Drawdown (%)'],
        'FinalCap': metrics_wf['Final Capital ($)']
    })
    cumulative_equity.extend(equity_wf[1:])

rolling_df = pd.DataFrame(rolling_summary)
print("\n" + "="*85)
print(f"{'Season':<10} | {'Bets':<5} | {'Win Rate':<10} | {'ROI (%)':<10} | {'Max DD (%)':<10} | {'Final Cap'}")
print("-" * 85)
for _, row in rolling_df.iterrows():
    print(f"{int(row['Season']):<10} | {int(row['Bets']):<5} | {row['WinRate']:<9.1f}% | {row['ROI']:<9.1f}% | -{row['MDD']:<9.1f}% | ${row['FinalCap']:.2f}")
print("="*85)

In [ ]:
# ==========================================
# Step 11.5: results visualization with enhanced aesthetics and dynamic labeling
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 6))

colors = ['#FFA131' if r > 0 else '#EF3435' for r in rolling_df['ROI']]
bars = ax1.bar(rolling_df['Season'], rolling_df['ROI'], color=colors, alpha=0.7, label='Yearly ROI (%)', zorder=2)
ax1.set_ylabel('ROI (%)', fontsize=12, fontweight='bold')
ax1.axhline(0, color='black', linewidth=1.2, zorder=3) 


ax2 = ax1.twinx()
ax2.plot(rolling_df['Season'], rolling_df['FinalCap'], color='#5656E6', marker='o', linewidth=3, markersize=8, label='Cumulative Capital ($)', zorder=4)
ax2.set_ylabel('Final Capital ($)', fontsize=12, fontweight='bold', color='#5656E6')
ax2.tick_params(axis='y', labelcolor='#5656E6')


for bar in bars:
    yval = bar.get_height()
    va_type = 'bottom' if yval > 0 else 'top'
    offset = 0.5 if yval > 0 else -0.5
    ax1.text(bar.get_x() + bar.get_width()/2, yval + offset, f"{yval:.1f}%", 
             ha='center', va=va_type, fontweight='bold', fontsize=11, zorder=5)

ax1.grid(False)
ax2.grid(False)
fig.tight_layout()
plt.savefig('step10_optimized_rolling.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# Exp A: Historical Regime Distribution Analysis
# ==========================================
print("\n[Experiment A] Analyzing Historical Regime Distributions...")

# Extract match results during the rolling test period
actual_results = df[df['Date'].dt.year.isin(test_years)].copy()
actual_results['Year'] = actual_results['Date'].dt.year

# Calculate the percentage of H/D/A for each year
regime_stats = actual_results.groupby(['Year', 'FTR']).size().unstack(fill_value=0)
regime_dist = regime_stats.div(regime_stats.sum(axis=1), axis=0) * 100

# Plot the distribution
ax = regime_dist.plot(kind='bar', stacked=True, figsize=(10, 6), 
                      color=['#EF3435', '#FFA131', '#5656E6'], alpha=0.8)


plt.ylabel('Percentage of Matches (%)')
plt.xlabel('Season')
plt.legend(['Away Win (A)', 'Draw (D)', 'Home Win (H)'], loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(False)

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy() 
    if height > 5: # only label if the segment is large enough to avoid clutter
        ax.text(x + width/2, y + height/2, f'{height:.1f}%', 
                ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('experiment_a_regime_dist.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# Experiment B: Model Confidence Interval ROI Analysis
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def get_col(candidates, df):
    for c in candidates:
        if c in df.columns: return c
    return None

prob_col = get_col(['Prob_LogicA', 'Pick_Prob', 'Prob'], bt_df)
pick_col = get_col(['Pick_LogicA', 'Pick'], bt_df)
odds_col = get_col(['Avg_Odds', 'Max_Odds'], bt_df)

if not prob_col or not pick_col:
    print("error: required columns for probability or pick not found in DataFrame.")
else:
    if not odds_col:
        bt_df['Temp_Odds'] = bt_df.apply(lambda row: row[f'Avg{row[pick_col]}'] if f'Avg{row[pick_col]}' in bt_df.columns else 1.0, axis=1)
        odds_col = 'Temp_Odds'

    bt_df['PnL'] = np.where(bt_df[pick_col] == bt_df['FTR'], bt_df[odds_col] - 1, -1)

    bins = [0.3, 0.45, 0.6, 1.0]
    labels = ['Low (0.3-0.45)', 'Mid (0.45-0.6)', 'High (0.6-1.0)']
    bt_df['Prob_Bin'] = pd.cut(bt_df[prob_col], bins=bins, labels=labels)

    bin_stats = bt_df.groupby('Prob_Bin', observed=True).agg(
        Bets=('PnL', 'count'),
        Total_PnL=('PnL', 'sum')
    ).reset_index()
    bin_stats['ROI'] = (bin_stats['Total_PnL'] / bin_stats['Bets']) * 100


plt.figure(figsize=(12, 7))
sns.set_style("white") 

colors = ['#5656E6' if r > 0 else '#EF3435' for r in bin_stats['ROI']]

ax = sns.barplot(data=bin_stats, x='Prob_Bin', y='ROI', palette=colors, alpha=0.8, zorder=2)

plt.axhline(0, color='black', linewidth=1.5, zorder=3)


plt.ylabel('ROI (%)', fontsize=14, fontweight='bold')
plt.xlabel('Model Predicted Probability (Confidence Level)', fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

for i, bar in enumerate(ax.patches):
    height = bar.get_height()
    offset = 1.0 if height > 0 else -3.0 
    
    if 0 > height > -2: offset = -2.5
        
    ax.text(bar.get_x() + bar.get_width()/2, height + offset, 
             f"ROI: {height:.1f}%\n(n={int(bin_stats.loc[i, 'Bets'])})", 
             ha='center', va='center', fontweight='bold', fontsize=11, color='black', zorder=5)

all_rois = bin_stats['ROI'].values
plt.ylim(min(all_rois) - 5, max(all_rois) + 5)
sns.despine()

plt.tight_layout()
plt.savefig('experiment_b_optimized.png', dpi=300, bbox_inches='tight')
plt.show()